In [44]:
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import tensorflow as tf
from keras import Sequential
from keras.layers import (GlobalAveragePooling2D,Dense)
import cv2
import numpy as np

In [45]:
image_shape = (224, 224, 3)
batch_size = 32

image_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=90,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,  # True if you plan to show your hand upside down
    fill_mode='nearest'
)

train_image_gen = image_gen.flow_from_directory(
    'train',
    target_size=image_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True  # enable shuffling to make the model more robust
)

test_image_gen = image_gen.flow_from_directory(
    'test',
    target_size=image_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # disable shuffling to keep data in same order as labels
)
validation_generator = image_gen.flow_from_directory(
        'val',
        target_size=image_shape[:2],
        batch_size=batch_size,
        class_mode='categorical',
        shuffle = False)  # disable shuffling to keep data in same order as labels

Found 1900 images belonging to 3 classes.
Found 272 images belonging to 3 classes.
Found 545 images belonging to 3 classes.


In [46]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

In [47]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Funct  (None, 7, 7, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d_6   (None, 1280)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_14 (Dense)            (None, 128)               163968    
                                                                 
 dropout_7 (Dropout)         (None, 128)               0         
                                                                 
 dense_15 (Dense)            (None, 3)                 387       
                                                                 
Total params: 2,422,339
Trainable params: 164,355
Non-

In [48]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [51]:
history = model.fit(
    train_image_gen,
    validation_data=validation_generator,
    epochs=20
)

Epoch 1/20
60/60 [==============================] - 43s 715ms/step - loss: 0.1733 - accuracy: 0.9358 - val_loss: 0.1009 - val_accuracy: 0.9633
Epoch 2/20
60/60 [==============================] - 42s 702ms/step - loss: 0.1737 - accuracy: 0.9337 - val_loss: 0.0916 - val_accuracy: 0.9725
Epoch 3/20
60/60 [==============================] - 44s 726ms/step - loss: 0.1565 - accuracy: 0.9468 - val_loss: 0.1059 - val_accuracy: 0.9651
Epoch 4/20
60/60 [==============================] - 44s 728ms/step - loss: 0.1377 - accuracy: 0.9526 - val_loss: 0.0920 - val_accuracy: 0.9688
Epoch 5/20
60/60 [==============================] - 45s 745ms/step - loss: 0.1340 - accuracy: 0.9516 - val_loss: 0.0689 - val_accuracy: 0.9706
Epoch 6/20
60/60 [==============================] - 42s 703ms/step - loss: 0.1571 - accuracy: 0.9447 - val_loss: 0.0962 - val_accuracy: 0.9615
Epoch 7/20
60/60 [==============================] - 43s 718ms/step - loss: 0.1399 - accuracy: 0.9516 - val_loss: 0.0623 - val_accuracy: 0.9817

In [52]:
loss, accuracy = model.evaluate(test_image_gen)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

9/9 [==============================] - 6s 601ms/step - loss: 0.1032 - accuracy: 0.9632
Test Loss: 0.10322584956884384
Test Accuracy: 0.9632353186607361


In [ ]:
# Replace these with your actual class names in the correct order
class_names = ["Paper", "Rock", "Scissor"]

# Image size used during training
IMG_SIZE = 224

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Convert BGR (OpenCV) to RGB (TensorFlow/Keras)
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Resize image
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Normalize
    img = img.astype("float32") / 255.0

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    # Predict
    pred = model.predict(img, verbose=0)

    # Get predicted class and confidence
    class_index = np.argmax(pred[0])
    confidence = pred[0][class_index] * 100

    # Print probabilities and class index
    # print("Probabilities:", pred[0])
    # print("Predicted class index:", class_index)

    # Create label
    label = f"{class_names[class_index]} ({confidence:.2f}%)"

    # Display label on frame
    cv2.putText(
        frame,
        label,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # Show webcam
    cv2.imshow("CNN Prediction", frame)

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

In [ ]:
model.save("rps.keras")